# Part 3 - Machine Learning: predict no-shows and choose the model honestly
**Project: CareLoop Clinics - No-Show Guard**

**Goal:** a risk score for every booking that the front desk can use **the evening before the
appointment** to decide whom to phone. At that moment everything we use as a feature is already
known: the patient's age, distance, fee, how long ago they booked, how many times they missed
before, and whether today's reminder was sent.

Only about 1 booking in 5 is a no-show, so accuracy alone is misleading. We judge every model
by **recall**, **precision** and **ROC-AUC** as well.

**The rule for this whole notebook:** the test set is for *reporting only*. We never use it to
choose a model, a setting or a threshold. All choices are made on the training set with
cross-validation.

| Step | What we do |
|---|---|
| 3.1 | Build the features, split 80/20, compute the baseline |
| 3.2 | Logistic regression (with scaling) and what its coefficients mean |
| 3.3 | Decision tree: unlimited depth vs depth 3 (overfitting) |
| 3.4 | Random forest and a comparison table |
| 3.5 | Tune, choose the model and the threshold with cross-validation, then test once |

In [1]:
# Install the libraries this notebook needs (safe to re-run).
%pip install -q --disable-pip-version-check pandas numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

# On Google Colab the project files are not there yet, so we download (clone) the repo first.
if "google.colab" in str(get_ipython()) and not Path("MASAI-CAPSTONE-PROJ").exists():
    os.system("git clone -q https://github.com/Advay4706/MASAI-CAPSTONE-PROJ")
if Path("MASAI-CAPSTONE-PROJ").exists():
    os.chdir("MASAI-CAPSTONE-PROJ")

BASE = Path(".") if Path("data").exists() else Path("..")   # the project's top folder
OUT = BASE / "outputs"

RANDOM_STATE = 42     # the same "seed" everywhere, so every run gives the same result
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

scikit-learn 1.8.0 | pandas 3.0.1 | numpy 2.4.3


## Step 3.1 - Features, split and baseline

**What:** turn the clean table into a feature table `X` (numbers only) and a target `y`
(`no_show`, 1 or 0). Split the rows 80% for training and 20% for testing. Work out the
*baseline*: the accuracy of a "model" that always says "Attended".

**Why each choice:**

- **Features used:** `age`, `distance_km`, `fee_inr`, `lead_days`, `prior_no_shows`, `reminder_sent`,
  `booking_channel`, `visit_type`.
- **Not used:** `appointment_id` and `patient_id` (labels, not information), the dates (already
  summarised by `lead_days`), `clinic_id` and `gender` (left out by the brief), and `status`
  (it *is* the answer - using it would be cheating, called **target leakage**).
- **`.map({"Yes": 1, "No": 0})`** turns a two-value text column into 1/0.
- **One-hot encoding** (`pd.get_dummies`) turns a text column with several values into one
  1/0 column per value. We cannot write App=1, Phone=2, Walk-In=3, because that would tell the
  model that Walk-In is "three times" App, which means nothing.
- **Train/test split:** the model learns from the training rows and is graded on test rows it
  has never seen. `random_state=42` makes the random split repeatable.
- We do **not** re-sort the rows: the split depends on the row order of the file.

In [3]:
from sklearn.model_selection import train_test_split

df = pd.read_csv(OUT / "appointments_clean.csv")        # keep the file's row order
print("Loaded:", df.shape)

feature_columns = ["age", "distance_km", "fee_inr", "lead_days", "prior_no_shows",
                   "reminder_sent", "booking_channel", "visit_type"]

X = df[feature_columns].copy()
X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})          # text -> 1/0
X = pd.get_dummies(X, columns=["booking_channel", "visit_type"])          # one-hot encoding
y = df["no_show"]                                                         # the target

print("X has", X.shape[1], "columns:")
print(list(X.columns))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

print("\nTraining rows:", len(X_train), "| test rows:", len(X_test))
print("No-shows in the test set:", int(y_test.sum()), f"({y_test.mean():.1%})")
print("No-shows in the training set:", int(y_train.sum()), f"({y_train.mean():.1%})")

# Baseline: always predict "Attended" (0). It is right for every patient who attended.
baseline_accuracy = (y_test == 0).mean()
print(f"\nBaseline accuracy (always predict 'Attended'): {baseline_accuracy:.4f} "
      f"= {(y_test == 0).sum()} of {len(y_test)}")

Loaded: (990, 16)
X has 11 columns:
['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']

Training rows: 792 | test rows: 198
No-shows in the test set: 49 (24.7%)
No-shows in the training set: 169 (21.3%)

Baseline accuracy (always predict 'Attended'): 0.7525 = 149 of 198


**Reading the result**

- `X` has **11 columns**: 6 numeric ones, 3 for `booking_channel` and 2 for `visit_type`.
- **792** training rows and **198** test rows; **49** of the test rows are no-shows.
- **Baseline accuracy is 0.7525** (149 of 198). A model that never flags anyone is 75% "accurate"
  and completely useless - it catches zero no-shows. This is why accuracy alone is a trap on
  **imbalanced** data, and why a real model must beat 0.7525 *and* have useful recall.